In [ ]:
import kagglehub
import pandas as pd
import os
from collections import Counter
from langdetect import detect
import random
import csv
import json
from sklearn.metrics import cohen_kappa_score

c:\Users\aryan\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Download the dataset
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
print("Path to dataset files:", path)

# Load the dataset into a dataframe
data = pd.read_csv(path + "/twcs/twcs.csv")

Path to dataset files: C:\Users\aryan\.cache\kagglehub\datasets\thoughtvector\customer-support-on-twitter\versions\10


In [3]:
data.head(20)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0
5,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,@115712 Can you please send us a private messa...,"5,7",8.0
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,sprintcare,False,Tue Oct 31 22:10:35 +0000 2017,@115713 This is saddening to hear. Please shoo...,NaN,12.0
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,"11,13,14",15.0
9,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,@115713 We understand your concerns and we'd l...,12,16.0


From the data I have understood that

1. Inbound refers to tweets that was recieved by companies, hence True and outbond are those tweets which are tweeted by companies in response to users, which is True.

2. If in_response_to_tweet_id is Nan, that means those were the root threads.

3. If response_tweet_id is Nan, that means those were the last threads in the conversation.

4. Inorder to make underatsnding more eay, we need to map all those threads together.

In [4]:
print("Shape of the data is: ", data.shape)
print("Columns in the data are: ", data.columns.tolist())
print(data.info())

Shape of the data is:  (2811774, 7)
Columns in the data are:  ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB
None


Convert in_response_tweet_id to Int64, created_at to date time format and response_tweet_id into a list

In [5]:
data["created_at"] = pd.to_datetime(data['created_at'])
data["in_response_to_tweet_id"] = data["in_response_to_tweet_id"].astype("Int64")

def parseResponseTweetIds(val):
    if pd.isna(val):
        return []
    return [int(x) for x in str(val).split(",")]

data["response_tweet_id"] = data["response_tweet_id"].apply(parseResponseTweetIds)

C:\Users\aryan\AppData\Local\Temp\ipykernel_14116\1826347718.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["created_at"] = pd.to_datetime(data['created_at'])


In [6]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype              
---  ------                   -----              
 0   tweet_id                 int64              
 1   author_id                object             
 2   inbound                  bool               
 3   created_at               datetime64[ns, UTC]
 4   text                     object             
 5   response_tweet_id        object             
 6   in_response_to_tweet_id  Int64              
dtypes: Int64(1), bool(1), datetime64[ns, UTC](1), int64(1), object(3)
memory usage: 134.1+ MB
None


In [7]:
# Find the number of times author_id appears in the data
print("Number of times author_id appears in the data is: ", data['author_id'].value_counts())

Number of times author_id appears in the data is:  author_id
AmazonHelp      169840
AppleSupport    106860
Uber_Support     56270
SpotifyCares     43265
Delta            42253
                 ...  
456282               1
456281               1
456280               1
456276               1
823870               1
Name: count, Length: 702777, dtype: int64


As AmazonHelp has the highest volume of tweets, I decided with go with the brand "AmazonHelp"

Also these are just the tweets that have been sent by AmazonHelp, we need to get all the tweets in the thread for a better context.

In [8]:
tweet_index = data.set_index('tweet_id')

roots = data[data['in_response_to_tweet_id'].isna()]['tweet_id'].tolist()

def collectThreads(rootId):
    threadTweets = []
    frontier = [rootId]
    seen = set()
    while frontier:
        nextFrontier = []

        for tid in frontier:
            if tid in seen or tid not in tweet_index.index:
                continue

            seen.add(tid)

            row = tweet_index.loc[tid]
            threadTweets.append({
                "tweet_id": tid,
                "author_id": row['author_id'],
                "inbound": row['inbound'],
                "text": row["text"],
                "created_at": row["created_at"],
            })
            children = row["response_tweet_id"]
            if children:
                nextFrontier.extend(children)
        frontier = nextFrontier
    threadTweets.sort(key=lambda x: x["created_at"])
    return threadTweets

amazonThreads = []
for rootId in roots:
    tweets = collectThreads(rootId)
    if any(t['author_id'] == 'AmazonHelp' for t in tweets):
        amazonThreads.append({"thread_id": str(rootId), "tweets": tweets})

print(f"{len(amazonThreads)} Amazon threads found in sample of {len(roots)} roots")


81902 Amazon threads found in sample of 794335 roots


In [9]:
# Checking the languages used in the threads
def detectLanguage(text):
    try:
        return detect(text)
    except:
        return "unknown"

langs = [detectLanguage(t["tweets"][0]["text"]) for t in amazonThreads]
langCounts = Counter(langs)

total = len(langs)

for lang, count in langCounts.most_common(15):
    print(f"{lang}: {count} ({count/total*100:.1f}%)")

en: 61273 (74.8%)
ja: 6522 (8.0%)
es: 4160 (5.1%)
fr: 3290 (4.0%)
de: 2526 (3.1%)
pt: 1620 (2.0%)
it: 925 (1.1%)
nl: 309 (0.4%)
hu: 255 (0.3%)
unknown: 124 (0.2%)
da: 107 (0.1%)
ca: 76 (0.1%)
so: 71 (0.1%)
tl: 64 (0.1%)
hi: 58 (0.1%)


So we can see that around 75% of the threads are in English and the rest 25% are a long tail of different languages. I would scope this project to English only and just train the agent on English language and make it a single language agent rather than multi language agent.

In [10]:
# Considering only English threads for further analysis
amazonThreads = [t for t in amazonThreads if detectLanguage(t["tweets"][0]["text"]) == "en"]

In [11]:
import random

def getOpeningText(thread):
    first = thread["tweets"][0]
    if not first["inbound"]:
        return None
    return first["text"]
    
random.seed(42)

sampleForReading = random.sample(amazonThreads, 150)

for t in sampleForReading:
    firstText = getOpeningText(t)
    print(f"[{t['thread_id']}] {firstText}")
    print("---")

[2096146] @AmazonHelp Can you give any information of when rainbow six siege th  video game will be restocked?
---
[319692] #KaroMilkeTayyari toh Diwali Diwali lagti hai. Amazon Great Indian Festival is back with great deals on your favourite brands. Stay tuned! https://t.co/n6PUnOnqPm
---
[98595] @115821 Why does your two day shipping say my RAM will arrive Tuesday? That is 5 days including Sunday.
---
[2428227] @115830 when they say your parcel has been delivered to resident only to find it in the garden waste with a dead rat, dead mice and birds that your cats have brought home, not a happy bunny at all 😡
---
[739085] @AmazonHelp Msg says tried 2 deliver @33104 last night &amp; just now...BUT am home (have cctv) but no courier...they must hv wrong house! help
---
[641661] @115821 what's the point of "prime shipping" when you don't deliver?!? Sadly you are off to a bad start this holiday. #saynotoprime #holidayshopping
---
[570008] a month @115821 cant verify police report for missin

In [12]:
# Saving the processed threads on the disk
import json
with open('../data/amazon_english_threads.json', 'w') as f:
    json.dump(amazonThreads, f, default=str)

So now to build the hand labeled 250 labelled examples, I sampled 300 threads at random from the English filtered AmazonHelp thread set ('random.seed(7)'), oversampling because some threads might be dropped as off-topic.

So I will label each row by hand and not by a LLM with:
- `intent` — one of the 8 taxonomy categories, or `off_topic` if the message isn't a genuine support request
- `auto_or_escalate` — my judgment on whether this should be auto-handled or escalated to a human
- `reason` — a short justification for that decision

In [ ]:
with open('../data/amazon_english_threads.json', encoding='utf-8') as f:
    threads = json.load(f)

random.seed(7)
golden_candidates = random.sample(threads, 300)

def get_customer_and_reply(thread):
    tweets = thread['tweets']
    customer_msg = tweets[0]['text'] if tweets else ''
    amazon_reply = next((t['text'] for t in tweets if t['author_id'] == 'AmazonHelp'), '')
    return customer_msg, amazon_reply

os.makedirs('../eval', exist_ok=True)

rows = []
for t in golden_candidates:
    customer_msg, amazon_reply = get_customer_and_reply(t)
    rows.append({
        'thread_id': t['thread_id'],
        'customer_message': customer_msg,
        'amazon_reply': amazon_reply,
        'intent': '',
        'auto_or_escalate': '',
        'reason': '',
    })

with open('../eval/golden_set_template.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['thread_id', 'customer_message', 'amazon_reply', 'intent', 'auto_or_escalate', 'reason'])
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} clean UTF-8 rows to eval/golden_set_template.csv")

Wrote 300 clean UTF-8 rows to eval/golden_set_template.csv


In [12]:
INPUT = '../eval/golden_set_template.csv'
OUTPUT = '../eval/golden_set_clean.csv'
EXPECTED_FIELDS = ['thread_id', 'customer_message', 'amazon_reply', 'intent', 'auto_or_escalate', 'reason']

with open(INPUT, encoding='utf-8', newline='') as f:
    reader = csv.reader(f)
    header = next(reader)
    
    fixed_rows = []
    for i, row in enumerate(reader, start=2):
        if len(row) > len(EXPECTED_FIELDS):
            extra = row[len(EXPECTED_FIELDS):]
            if all(e.strip() == '' for e in extra):
                row = row[:len(EXPECTED_FIELDS)]
            else:
                print(f"WARNING: row {i} has non-empty extra fields, needs manual check: {row}")
                continue
        elif len(row) < len(EXPECTED_FIELDS):
            print(f"WARNING: row {i} has too FEW fields, needs manual check: {row}")
            continue
        fixed_rows.append(row)

with open(OUTPUT, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(EXPECTED_FIELDS)
    writer.writerows(fixed_rows)

print(f"Wrote {len(fixed_rows)} clean rows to {OUTPUT}")

Wrote 300 clean rows to ../eval/golden_set_clean.csv


In [5]:
df = pd.read_csv('../eval/golden_set_clean.csv', encoding='utf-8')
print(df.shape)
print(df.columns.tolist())
print(df['intent'].value_counts())

(300, 6)
['thread_id', 'customer_message', 'amazon_reply', 'intent', 'auto_or_escalate', 'reason']
intent
general_complaint        36
informational            27
delivery_delay           19
delivery_not_recieved    19
account_security         15
refund_billing           11
item_quality_issue       10
off_topic                 7
app_device_technical      6
informaional              5
delivery_not_received     2
delivery_not_recived      1
general_comaplint         1
item_quality              1
Name: count, dtype: int64


As they were manually typed there seems to be a lot of typing errors, so we will fix that

In [6]:
intent_fixes = {
    'delivery_not_recieved': 'delivery_not_received',
    'delivery_not_recived': 'delivery_not_received',
    'informaional': 'informational',
    'general_comaplint': 'general_complaint',
    'item_quality': 'item_quality_issue',
}

df['intent'] = df['intent'].replace(intent_fixes)
print(df['intent'].value_counts())

intent
general_complaint        37
informational            32
delivery_not_received    22
delivery_delay           19
account_security         15
item_quality_issue       11
refund_billing           11
off_topic                 7
app_device_technical      6
Name: count, dtype: int64


In [7]:
print(df['auto_or_escalate'].value_counts())

auto_or_escalate
auto         82
escalate     67
esacalate     3
auo           1
Name: count, dtype: int64


In [8]:
decision_fixes = {
    'esacalate': 'escalate',
    'auo': 'auto',
}
df['auto_or_escalate'] = df['auto_or_escalate'].replace(decision_fixes)
print(df['auto_or_escalate'].value_counts())

auto_or_escalate
auto        83
escalate    70
Name: count, dtype: int64


In [ ]:
df_labeled = df[df['intent'].notna() & (df['intent'].str.strip() != '')].copy()
print(f"\nLabeled rows: {df_labeled.shape}")

df_labeled.to_csv('../eval/golden_set.csv', index=False, encoding='utf-8')
print("Saved ../eval/golden_set.csv")


Labeled rows: (160, 6)
Saved eval/golden_set.csv


In [ ]:
df = pd.read_csv('../eval/judge_sample.csv', encoding='utf-8')

for dim in ['relevance', 'groundedness', 'tone']:
    judge_col = f'judge_{dim}'
    human_col = f'human_{dim}'
    
    exact_match = (df[judge_col] == df[human_col]).mean()
    within_one = (abs(df[judge_col] - df[human_col]) <= 1).mean()
    
    print(f"{dim}: exact match = {exact_match:.2f}, within 1 point = {within_one:.2f}")

relevance: exact match = 0.23, within 1 point = 0.63
groundedness: exact match = 0.33, within 1 point = 0.47
tone: exact match = 0.73, within 1 point = 0.97
